 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           :                 <br>
Student Number : L XXXXX         <br>
Due Date       :                 <br>
Assignment     : CA2             <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [190]:
# pip installs
!pip install pytesseract pdf2image PyPDF2 pillow spacy nltk scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [191]:
#download spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 30.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [192]:
# imports
# imports
import sys
print(sys.executable)
import pymupdf

import site
print(site.getusersitepackages())

import matplotlib.pyplot as plt
# allows inline plotting
%matplotlib inline

import pytesseract
from pytesseract import Output
# regular expression
import re

from pdf2image import convert_from_path

import PyPDF2

import os
import cv2




import pandas as pd
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

# load spaCy model
nlp = spacy.load("en_core_web_sm")

from PIL import Image


/Users/shanegannon/CA2_Assignment/.venv/bin/python
/Users/shanegannon/Library/Python/3.13/lib/python/site-packages


# Support Functions

In [193]:
# code here

import PyPDF2
import pytesseract
from pdf2image import convert_from_path

#function to set up tesseract
def setup_tesseract(tesseract_path="/opt/homebrew/bin/tesseract", oem=1, psm=6):
    pytesseract.pytesseract.tesseract_cmd = tesseract_path
    custom_config = f"--oem {oem} --psm {psm}"
    return custom_config


def pdf_to_images(pdf_path, poppler_path="/opt/homebrew/bin"):
    pages = convert_from_path(pdf_path, poppler_path=poppler_path)
    return pages


def extract_text_from_image(img, config):
    text = pytesseract.image_to_string(img, config=config)
    return text


def extract_text_from_pdf_direct(pdf_path):
    all_text = []

    with open(pdf_path, "rb") as pdf_file:
        pdf_reader = PyPDF2.PdfReader(pdf_file)

        for page_number, page in enumerate(pdf_reader.pages, start=1):
            page_text = page.extract_text()

            all_text.append({
                "page": page_number,
                "text": page_text if page_text else ""
            })

    return all_text


def extract_text_from_pdf_ocr(pdf_path,
                              poppler_path="/opt/homebrew/bin",
                              tesseract_path="/opt/homebrew/bin/tesseract",
                              oem=1,
                              psm=6):
    config = setup_tesseract(
        tesseract_path=tesseract_path,
        oem=oem,
        psm=psm
    )

    pages = pdf_to_images(pdf_path, poppler_path=poppler_path)
    all_text = []

    for page_number, img in enumerate(pages, start=1):
        page_text = extract_text_from_image(img, config)

        all_text.append({
            "page": page_number,
            "text": page_text
        })

    return all_text


def extract_text_from_pdf(pdf_path,
                          poppler_path="/opt/homebrew/bin",
                          tesseract_path="/opt/homebrew/bin/tesseract",
                          oem=1,
                          psm=6,
                          min_text_length=100):
    direct_text_output = extract_text_from_pdf_direct(pdf_path)

    full_direct_text = " ".join(page_data["text"] for page_data in direct_text_output)

    if full_direct_text and len(full_direct_text.strip()) > min_text_length:
        print("Using direct PDF extraction")
        return direct_text_output, "direct_pdf"

    print("Direct extraction was poor or empty, using OCR fallback")

    ocr_text_output = extract_text_from_pdf_ocr(
        pdf_path=pdf_path,
        poppler_path=poppler_path,
        tesseract_path=tesseract_path,
        oem=oem,
        psm=psm
    )

    return ocr_text_output, "ocr_pdf"

# return the file type used
def detect_file_type(file_path):
    file_ext = os.path.splitext(file_path)[1].lower()

    if file_ext == ".pdf":
        return "pdf"
    elif file_ext in [".jpg", ".jpeg", ".png"]:
        return "image"
    else:
        return "unsupported"
#Tesseract helper functions


#helper function to extract text
def extract_text_from_document(file_path,
                               poppler_path="/opt/homebrew/bin",
                               tesseract_path="/opt/homebrew/bin/tesseract",
                               oem=1,
                               psm=6,
                               min_text_length=100):
    file_type = detect_file_type(file_path)

    if file_type == "pdf":
        text_output, extraction_method = extract_text_from_pdf(
            pdf_path=file_path,
            poppler_path=poppler_path,
            tesseract_path=tesseract_path,
            oem=oem,
            psm=psm,
            min_text_length=min_text_length
        )
        return text_output, extraction_method

    elif file_type == "image":
        config = setup_tesseract(
            tesseract_path=tesseract_path,
            oem=oem,
            psm=psm
        )

        img = Image.open(file_path)
        image_text = extract_text_from_image(img, config)

        text_output = [{
            "page": 1,
            "text": image_text
        }]

        return text_output, "ocr_image"

    else:
        raise ValueError("Unsupported file type")


def detect_file_type(file_path):
    file_path = file_path.lower()

    if file_path.endswith(".pdf"):
        return "pdf"
    elif file_path.endswith((".png", ".jpg", ".jpeg", ".tiff", ".bmp")):
        return "image"
    else:
        return "unsupported"



    # text celaning helper methods
# Function to clean raw text
def clean_text(text):
    # Change to lowercase
    text = text.lower()

    # Remove extra spaces and line breaks
    text = re.sub(r"\s+", " ", text)

    # Remove unwanted characters but keep basic punctuation for now
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\;\:\-]", "", text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text


# Function to tokenize text using spaCy
def tokenize_text(text):
    doc_object = nlp(text)

    token_list = []

    for token in doc_object:
        token_list.append(token.text)

    return token_list


# Function to remove stop words using spaCy stop words
def remove_stopwords(text):
    doc_object = nlp(text)

    filtered_tokens = []

    for token in doc_object:
        if not token.is_stop and not token.is_punct and not token.is_space:
            filtered_tokens.append(token.text)

    return filtered_tokens


# Function to lemmatise text using spaCy
def lemmatise_text(text):
    doc_object = nlp(text)

    lemmatised_tokens = []

    for token in doc_object:
        if not token.is_stop and not token.is_punct and not token.is_space:
            lemmatised_tokens.append(token.lemma_)

    return lemmatised_tokens


# Function to convert token list back into a string
def tokens_to_string(token_list):
    text_string = " ".join(token_list)
    return text_string


# Function to count words in a text string
def count_words(text):
    word_total = len(text.split())
    return word_total


# Function to extract named entities from text
def extract_named_entities(text):
    doc_object = nlp(text)

    entity_list = []

    for ent in doc_object.ents:
        entity_list.append((ent.text, ent.label_))

    return entity_list


# Function to create a processed record for one document/page
def process_document_text(document_id, file_name, page_number, extraction_method, raw_text):
    cleaned_text = clean_text(raw_text)

    tokenised_tokens = tokenize_text(cleaned_text)
    tokenised_text = tokens_to_string(tokenised_tokens)

    no_stop_tokens = remove_stopwords(cleaned_text)
    no_stop_text = tokens_to_string(no_stop_tokens)

    lemmatised_tokens = lemmatise_text(cleaned_text)
    lemmatised_text = tokens_to_string(lemmatised_tokens)

    named_entities = extract_named_entities(cleaned_text)

    document_record = {
        "document_id": document_id,
        "file_name": file_name,
        "page_number": page_number,
        "extraction_method": extraction_method,
        "raw_text": raw_text,
        "cleaned_text": cleaned_text,
        "tokenised_text": tokenised_text,
        "no_stop_text": no_stop_text,
        "lemmatised_text": lemmatised_text,
        "raw_word_count": count_words(raw_text),
        "cleaned_word_count": count_words(cleaned_text),
        "named_entities": named_entities
    }

    return document_record


# Function to create a dataframe from extracted document text output
def build_document_dataframe(extracted_text_list, file_name, extraction_method="unknown", document_id=1):
    document_records = []

    for page_data in extracted_text_list:
        page_number = page_data["page"]
        raw_text = page_data["text"]

        processed_record = process_document_text(
            document_id=document_id,
            file_name=file_name,
            page_number=page_number,
            extraction_method=extraction_method,
            raw_text=raw_text
        )

        document_records.append(processed_record)

    documents_df = pd.DataFrame(document_records)

    return documents_df




#add a dpcument to the data frame given a path
def add_document_to_dataframe(documents_df, file_path, document_id,
                              poppler_path="/opt/homebrew/bin",
                              tesseract_path="/opt/homebrew/bin/tesseract",
                              oem=1,
                              psm=6,
                              min_text_length=100):
    text_output, extraction_method = extract_text_from_document(
        file_path=file_path,
        poppler_path=poppler_path,
        tesseract_path=tesseract_path,
        oem=oem,
        psm=psm,
        min_text_length=min_text_length
    )

    for page_data in text_output:
        documents_df = add_to_dataframe(
            documents_df=documents_df,
            document_id=document_id,
            file_name=file_path,
            page_number=page_data["page"],
            extraction_method=extraction_method,
            raw_text=page_data["text"]
        )

    return documents_df


#printer helper method
def print_document_row(documents_df, row_index):
    row_data = documents_df.iloc[row_index]

    print(f"Description: Document ID")
    print(row_data["document_id"])
    print()

    print(f"Description: File name")
    print(row_data["file_name"])
    print()

    print(f"Description: Page number")
    print(row_data["page_number"])
    print()

    print(f"Description: Extraction method")
    print(row_data["extraction_method"])
    print()

    print(f"Description: Raw text")
    print(row_data["raw_text"])
    print()

    print(f"Description: Cleaned text")
    print(row_data["cleaned_text"])
    print()

    print(f"Description: Tokenised text")
    print(row_data["tokenised_text"])
    print()

    print(f"Description: Text with stop words removed")
    print(row_data["no_stop_text"])
    print()

    print(f"Description: Lemmatised text")
    print(row_data["lemmatised_text"])
    print()

    print(f"Description: Raw word count")
    print(row_data["raw_word_count"])
    print()

    print(f"Description: Cleaned word count")
    print(row_data["cleaned_word_count"])
    print()

    print(f"Description: Named entities")
    print(row_data["named_entities"])


# NLP

## Text extraction

In [201]:
## code

documents_df = pd.DataFrame(columns=[
    "document_id",
    "file_name",
    "page_number",
    "extraction_method",
    "raw_text",
    "cleaned_text",
    "tokenised_text",
    "no_stop_text",
    "lemmatised_text",
    "raw_word_count",
    "cleaned_word_count",
    "named_entities"
])



# file_path = "sample.pdf"
#file_path = "A_Midsummer_Night.pdf"
#file_path = "HistoricalDoc2.jpg"
file_path = "85629964.png"

document_id = 1

documents_df = add_document_to_dataframe(
    documents_df,
    file_path,
    1
)

print_document_row(documents_df, 0)


Description: Document ID
1

Description: File name
85629964.png

Description: Page number
1

Description: Extraction method
ocr_image

Description: Raw text
a c c
CIGARETTE REPORT FORM

YEAR: NO. PER PACK:

BRAND NAME:

VAR. DESC: (SEE EXPLANATION)

VARIETY UNIT SALES: VARIETY DOLLAR SALES:

CIG. LENGTH: FILTER LENGTH:

FILTER TYPE: FLAVORING: OVERWRAP: PACK TYPE:

AST MANUFACT. DATE: 1ST SALES DATE: LAST SOLD DATE:

YEARLY SUMMARY:

TAR: NICOTINE: CARBON MONO:

ADVERTISING EXPENDITURES (SEE EXPLANATION)

CAT~A-EXPENSES: CAT-B-EXPENSES : CAT-C-EXPENSES:

CAT-D-EXPENSES: CAT-E-EXPENSES: CAT-P-EXPENSES:

CAT-G-EXPENSES: CAT-H-EXPENSES : CAT-1-EXPENSES :

CAT-J-EXPENSES: CAT-K-EXPENSES: CAT-L-EXPENSES:

CAT-M-EXPENSES : CAT-N-EXPENSES :

TOTAL ADVERTISING EXPENDITURES:
x
a
EA
8
3
3
cd
&

i


Description: Cleaned text
a c c cigarette report form year: no. per pack: brand name: var. desc: see explanation variety unit sales: variety dollar sales: cig. length: filter length: filter type: flav

In [198]:
print(documents_df[["document_id", "file_name", "page_number", "extraction_method"]].head(10))

   document_id              file_name  page_number extraction_method
0            1  A_Midsummer_Night.pdf            1        direct_pdf
1            1  A_Midsummer_Night.pdf            2        direct_pdf
2            1  A_Midsummer_Night.pdf            3        direct_pdf
3            1  A_Midsummer_Night.pdf            4        direct_pdf
4            1  A_Midsummer_Night.pdf            5        direct_pdf
5            1  A_Midsummer_Night.pdf            6        direct_pdf
6            1  A_Midsummer_Night.pdf            7        direct_pdf
7            1  A_Midsummer_Night.pdf            8        direct_pdf
8            1  A_Midsummer_Night.pdf            9        direct_pdf
9            1  A_Midsummer_Night.pdf           10        direct_pdf


# Vision

## Sub Heading 1

In [195]:
# code here...

# Multi-modal

## Sub Heading 1

In [196]:
# code here

# Final Output

In [197]:
# code